# Experiment A: ELI5 Constrained Decoding on CUDA

Run this notebook with a CUDA-enabled Python kernel in VS Code/Jupyter. The notebook uses the existing modular pipeline under `eli5_test`, but executes it inside the GPU kernel instead of the terminal interpreter.

Workflow:
1. Verify CUDA in the active kernel.
2. Load Qwen in 4-bit with GPU support.
3. Move inputs to CUDA and sanity-check generation.
4. Run generation for 100 ELI5 samples across all tiers.
5. Evaluate the saved generations in a separate step.
6. Handle CUDA and memory errors with clear diagnostics.

## 1. Verify CUDA in the Active Notebook Kernel

Confirm that this notebook is running on a CUDA-enabled PyTorch build before loading the model.

In [ ]:
from pathlib import Path
import sys
import torch


def find_eli5_test_dir(start_path: Path | None = None) -> Path:
    current = Path.cwd() if start_path is None else start_path
    if current.name == "eli5_test":
        return current
    for candidate in [current, *current.parents]:
        candidate_dir = candidate / "eli5_test"
        if candidate_dir.exists() and candidate_dir.is_dir():
            return candidate_dir
    raise RuntimeError("Could not locate eli5_test from the current notebook working directory.")


ELI5_TEST_DIR = find_eli5_test_dir()
eli5_path = str(ELI5_TEST_DIR)
if eli5_path not in sys.path:
    sys.path.insert(0, eli5_path)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"eli5_test dir: {ELI5_TEST_DIR}")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook must be attached to a CUDA-enabled kernel.")

device_name = torch.cuda.get_device_name(0)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"Active GPU: {device_name}")
print(f"Free VRAM: {free_bytes / 1024**3:.2f} GiB")
print(f"Total VRAM: {total_bytes / 1024**3:.2f} GiB")

PyTorch version: 2.5.1+cu121
CUDA build: 12.1
CUDA available: True
eli5_test dir: c:\Users\vimal\OneDrive\Documents\Uni\BTP\User-Adaptive-XAI\eli5_test
Active GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Free VRAM: 4.96 GiB
Total VRAM: 6.00 GiB


: 

## 2. Load Qwen with GPU Support

Load the model inside the notebook kernel using 4-bit quantization and the local constrained-decoding wrapper.

In [ ]:
from experiment_config import CONFIG
from model_loader import load_qwen_model, build_cd_generator

print(f"Model name: {CONFIG['model_name']}")
tokenizer, model = load_qwen_model()
generator = build_cd_generator(model=model, tokenizer=tokenizer)
print(f"Model device: {model.device}")
print(f"Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GiB")

c:\Users\vimal\gpu_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Move Tokenized Inputs and Model to CUDA

Sanity-check that tokenized inputs and model outputs are on the same GPU device before running the full experiment.

In [ ]:
sample_prompt = "Explain why the sky looks blue in simple terms."
sample_inputs = tokenizer(sample_prompt, return_tensors="pt").to(model.device)
print({key: value.device for key, value in sample_inputs.items()})

with torch.no_grad():
    sample_output = model.generate(**sample_inputs, max_new_tokens=32, do_sample=False)

print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

{'input_ids': device(type='cuda', index=0), 'attention_mask': device(type='cuda', index=0)}
Explain why the sky looks blue in simple terms. The sky appears blue because of a phenomenon called Rayleigh scattering. When sunlight enters our atmosphere, it encounters many tiny particles such as air molecules and dust particles.


## 4. Handle CUDA and Memory Errors

Use these helpers during generation to print VRAM diagnostics and recover cleanly from out-of-memory errors.

In [ ]:
def cuda_diagnostics() -> None:
    if not torch.cuda.is_available():
        print("CUDA is not available in this notebook kernel.")
        return
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Free VRAM: {free_bytes / 1024**3:.2f} GiB")
    print(f"Total VRAM: {total_bytes / 1024**3:.2f} GiB")


def safe_generate(system_prompt: str, task_prompt: str, user_category: str):
    try:
        return generator.generate(
            system_prompt=system_prompt,
            task_prompt=task_prompt,
            user_category=user_category,
        )
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print("[OOM] CUDA ran out of memory. Clearing cache and aborting this sample.")
            torch.cuda.empty_cache()
        raise


cuda_diagnostics()

GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Free VRAM: 3.81 GiB
Total VRAM: 6.00 GiB


## 5. Run Experiment A Generation on GPU

Generate answers for the first 100 ELI5 samples across the three tiers and save checkpointed raw outputs for later scoring.

In [ ]:
import gc
import pandas as pd
from tqdm.auto import tqdm

from utils import (
    ensure_parent_dir,
    extract_question,
    extract_reference_answer,
    load_eli5_samples,
    seed_everything,
)

seed_everything(CONFIG["seed"])

samples = load_eli5_samples(
    dataset_name=CONFIG["dataset_name"],
    split=CONFIG["dataset_split"],
    dataset_size=CONFIG["dataset_size"],
)

raw_rows = []
checkpoint_path = CONFIG["results"]["checkpoint"]
raw_results_path = CONFIG["results"]["raw"]
ensure_parent_dir(raw_results_path)

for sample_index, example in enumerate(tqdm(samples, desc="Samples", total=len(samples), position=0)):
    question = extract_question(example)
    reference_answer = extract_reference_answer(example)

    tier_iter = tqdm(
        CONFIG["tiers"].items(),
        desc=f"Sample {sample_index + 1}/{len(samples)}",
        total=len(CONFIG["tiers"]),
        leave=False,
        position=1,
    )

    for tier_name, tier_config in tier_iter:
        tier_iter.set_postfix_str(tier_name)
        try:
            generated_answer = safe_generate(
                system_prompt=tier_config["system_prompt"],
                task_prompt=question,
                user_category=tier_name,
            )
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print(f"[OOM] Sample {sample_index} / {tier_name}. Skipping this answer.")
                torch.cuda.empty_cache()
                generated_answer = ""
            else:
                raise

        raw_rows.append(
            {
                "sample_id": sample_index,
                "tier": tier_name,
                "question": question,
                "generated_answer": generated_answer,
                "reference_answer": reference_answer,
            }
        )

    del tier_iter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if (sample_index + 1) % CONFIG["checkpoint_interval"] == 0:
        pd.DataFrame(raw_rows).to_csv(checkpoint_path, index=False)
        print(f"[Checkpoint] Saved {len(raw_rows)} rows to {checkpoint_path}")

raw_df = pd.DataFrame(raw_rows)
raw_df.to_csv(raw_results_path, index=False)
raw_df.to_csv(checkpoint_path, index=False)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"[Done] Raw generation saved to {raw_results_path}")
print(f"[Done] Final checkpoint saved to {checkpoint_path}")

: 

## 6. Collect and Save Evaluation Outputs

Run the separate evaluation stage after generation has finished, then inspect the saved summary table.

In [ ]:
from evaluate_experiment_a import main as evaluate_main

evaluate_main()

summary_df = pd.read_csv(CONFIG["results"]["summary"])
summary_df